In [2]:
import numpy as np
import pandas as pd
from gensim.models import Word2Vec

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM
from tensorflow.keras.callbacks import EarlyStopping

In [3]:
train = pd.read_csv('../datasets/hwu/train.csv')
val = pd.read_csv('../datasets/hwu/val.csv')

In [4]:
print("Train shape:", train.shape)
print("Validation shape:", val.shape)
train.head()

Train shape: (8954, 2)
Validation shape: (1076, 2)


,text,category
0,what alarms do i have set right now,alarm_query
1,checkout today alarm of meeting,alarm_query
2,report alarm settings,alarm_query
3,see see for me the alarms that you have set to...,alarm_query
4,is there an alarm for ten am,alarm_query


In [5]:
le = LabelEncoder()
train_labels = le.fit_transform(train['category'])
val_labels = le.transform(val['category'])
num_classes = len(set(train_labels))
num_classes

64

### TF-IDF + Logistic Regression

In [6]:
tfidf_lr_pipeline = make_pipeline(
    TfidfVectorizer(max_features=5000),
    LogisticRegression(max_iter=1000)
)

In [7]:
tfidf_lr_pipeline.fit(train['text'], train_labels)

Pipeline(steps=[('tfidfvectorizer', TfidfVectorizer(max_features=5000)),
                ('logisticregression', LogisticRegression(max_iter=1000))])

In [8]:
y_pred = tfidf_lr_pipeline.predict(val['text'])
print(classification_report(val_labels, y_pred))

              precision    recall  f1-score   support

           0       0.78      0.95      0.86        19
           1       0.75      0.55      0.63        11
           2       0.74      0.74      0.74        19
           3       0.62      0.62      0.62         8
           4       1.00      0.53      0.70        15
           5       0.79      0.85      0.81        13
           6       0.77      0.53      0.62        19
           7       0.82      0.95      0.88        19
           8       0.88      0.79      0.83        19
           9       0.94      0.89      0.92        19
          10       0.56      0.62      0.59         8
          11       0.82      0.74      0.78        19
          12       1.00      0.88      0.93         8
          13       0.94      0.89      0.92        19
          14       0.94      0.84      0.89        19
          15       0.90      0.95      0.92        19
          16       1.00      0.89      0.94        19
          17       0.90    

c:\Users\LENOVO\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\LENOVO\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\LENOVO\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average,

### Word2Vec + Dense Layer

In [9]:
sentences = [text.split() for text in train['text']]
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4, epochs=50)

In [10]:
def sentence_to_avg_vector(text, model):
    words = text.split()
    vectors = []

    for w in words:
        if w in model.wv:
            vectors.append(model.wv[w])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [11]:
def convert_df_to_vectors(df, text_column, model):
    X = np.array([sentence_to_avg_vector(text, model)
                  for text in df[text_column]])
    return X

In [12]:
X_train_avg = convert_df_to_vectors(train, "text", w2v_model)
X_val_avg   = convert_df_to_vectors(val, "text", w2v_model)

In [13]:
dense_model = Sequential([
    Input((w2v_model.vector_size,)),
    Dense(256, activation='relu'),
    Dense(128, activation='relu'),
    Dense(num_classes, activation='softmax')
])

dense_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = dense_model.fit(X_train_avg, train_labels, epochs=20, batch_size=32, validation_data=(X_val_avg, val_labels))

Epoch 1/20
280/280 [==============================] - 1s 2ms/step - loss: 1.7983 - accuracy: 0.5539 - val_loss: 1.0333 - val_accuracy: 0.6952
Epoch 2/20
280/280 [==============================] - 0s 1ms/step - loss: 0.8923 - accuracy: 0.7431 - val_loss: 0.8817 - val_accuracy: 0.7351
Epoch 3/20
280/280 [==============================] - 0s 2ms/step - loss: 0.7696 - accuracy: 0.7778 - val_loss: 0.8353 - val_accuracy: 0.7714
Epoch 4/20
280/280 [==============================] - 0s 1ms/step - loss: 0.6920 - accuracy: 0.7947 - val_loss: 0.8037 - val_accuracy: 0.7584
Epoch 5/20
280/280 [==============================] - 0s 1ms/step - loss: 0.6348 - accuracy: 0.8151 - val_loss: 0.7299 - val_accuracy: 0.7881
Epoch 6/20
280/280 [==============================] - 0s 1ms/step - loss: 0.5896 - accuracy: 0.8273 - val_loss: 0.7547 - val_accuracy: 0.7853
Epoch 7/20
280/280 [==============================] - 0s 2ms/step - loss: 0.5532 - accuracy: 0.8308 - val_loss: 0.7686 - val_accuracy: 0.7890
Epoch 

In [14]:
y_pred = np.argmax(dense_model.predict(X_val_avg), axis=1)
print(classification_report(val_labels, y_pred))

34/34 [==============================] - 0s 653us/step
              precision    recall  f1-score   support

           0       0.90      0.95      0.92        19
           1       0.82      0.82      0.82        11
           2       0.74      0.74      0.74        19
           3       0.29      0.25      0.27         8
           4       1.00      0.73      0.85        15
           5       0.58      0.85      0.69        13
           6       0.48      0.58      0.52        19
           7       0.89      0.89      0.89        19
           8       0.73      0.58      0.65        19
           9       0.65      0.68      0.67        19
          10       0.75      0.75      0.75         8
          11       0.65      0.79      0.71        19
          12       1.00      0.62      0.77         8
          13       0.92      0.63      0.75        19
          14       0.87      0.68      0.76        19
          15       0.73      1.00      0.84        19
          16       0.90   

### Embedding Pre-trained + LSTM

In [15]:
tokenizer = Tokenizer(oov_token="<UNK>")
tokenizer.fit_on_texts(train['text'])
train_sequences = tokenizer.texts_to_sequences(train['text'])
val_sequences = tokenizer.texts_to_sequences(val['text'])

In [16]:
max_len = max(len(train_sequences[i]) for i in range(len(train_sequences)))
max_len

25

In [17]:
X_train_pad = pad_sequences(train_sequences, maxlen=max_len, padding='post', truncating='post')
X_val_pad = pad_sequences(val_sequences, maxlen=max_len, padding='post', truncating='post')

In [18]:
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = w2v_model.vector_size
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in tokenizer.word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]

In [19]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,         
    restore_best_weights=True
)

In [20]:
lstm_model_pretrained = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix], # Khởi tạo trọng số
        trainable=False # Đóng băng lớp Embedding
    ),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(num_classes, activation='softmax')
])

lstm_model_pretrained.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
history = lstm_model_pretrained.fit(X_train_pad, train_labels, epochs=20, validation_data=(X_val_pad, val_labels), batch_size=32, callbacks=[early_stop])

Epoch 1/20
280/280 [==============================] - 12s 35ms/step - loss: 2.7965 - accuracy: 0.2560 - val_loss: 1.7711 - val_accuracy: 0.5074
Epoch 2/20
280/280 [==============================] - 10s 37ms/step - loss: 1.6114 - accuracy: 0.5383 - val_loss: 1.2590 - val_accuracy: 0.6487
Epoch 3/20
280/280 [==============================] - 9s 34ms/step - loss: 1.2524 - accuracy: 0.6466 - val_loss: 1.1018 - val_accuracy: 0.7082
Epoch 4/20
280/280 [==============================] - 10s 35ms/step - loss: 1.0452 - accuracy: 0.7062 - val_loss: 0.9481 - val_accuracy: 0.7323
Epoch 5/20
280/280 [==============================] - 10s 35ms/step - loss: 0.9098 - accuracy: 0.7402 - val_loss: 0.8569 - val_accuracy: 0.7677
Epoch 6/20
280/280 [==============================] - 10s 34ms/step - loss: 0.8118 - accuracy: 0.7680 - val_loss: 0.7785 - val_accuracy: 0.7909
Epoch 7/20
280/280 [==============================] - 10s 37ms/step - loss: 0.7311 - accuracy: 0.7829 - val_loss: 0.7708 - val_accuracy: 

In [21]:
y_pred = np.argmax(lstm_model_pretrained.predict(X_val_pad), axis=1)
print(classification_report(val_labels, y_pred))

34/34 [==============================] - 0s 6ms/step
              precision    recall  f1-score   support

           0       0.90      0.95      0.92        19
           1       1.00      0.91      0.95        11
           2       0.89      0.84      0.86        19
           3       0.00      0.00      0.00         8
           4       0.75      0.60      0.67        15
           5       0.46      0.85      0.59        13
           6       0.75      0.63      0.69        19
           7       0.89      0.84      0.86        19
           8       0.80      0.84      0.82        19
           9       0.77      0.89      0.83        19
          10       1.00      0.88      0.93         8
          11       0.88      0.79      0.83        19
          12       1.00      0.88      0.93         8
          13       0.90      0.95      0.92        19
          14       0.86      0.63      0.73        19
          15       0.95      1.00      0.97        19
          16       0.90     

c:\Users\LENOVO\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\LENOVO\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\LENOVO\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average,

### Embedding + LSTM

In [22]:
lstm_model_scratch = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(num_classes, activation='softmax')
])

lstm_model_scratch.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
history = lstm_model_scratch.fit(X_train_pad, train_labels, epochs=20, validation_data=(X_val_pad, val_labels), batch_size=32, callbacks=[early_stop])

Epoch 1/20
280/280 [==============================] - 14s 44ms/step - loss: 3.8835 - accuracy: 0.0408 - val_loss: 3.2999 - val_accuracy: 0.0911
Epoch 2/20
280/280 [==============================] - 12s 44ms/step - loss: 3.0358 - accuracy: 0.1183 - val_loss: 2.7002 - val_accuracy: 0.1757
Epoch 3/20
280/280 [==============================] - 12s 43ms/step - loss: 2.4257 - accuracy: 0.2389 - val_loss: 2.1506 - val_accuracy: 0.3243
Epoch 4/20
280/280 [==============================] - 14s 51ms/step - loss: 1.7800 - accuracy: 0.4194 - val_loss: 1.6998 - val_accuracy: 0.4712
Epoch 5/20
280/280 [==============================] - 14s 48ms/step - loss: 1.2741 - accuracy: 0.6002 - val_loss: 1.2732 - val_accuracy: 0.6227
Epoch 6/20
280/280 [==============================] - 14s 51ms/step - loss: 0.9614 - accuracy: 0.7097 - val_loss: 1.1767 - val_accuracy: 0.6914
Epoch 7/20
280/280 [==============================] - 13s 46ms/step - loss: 0.7305 - accuracy: 0.7899 - val_loss: 1.0517 - val_accuracy:

In [23]:
y_pred = np.argmax(lstm_model_scratch.predict(X_val_pad), axis=1)
print(classification_report(val_labels, y_pred))

34/34 [==============================] - 0s 5ms/step
              precision    recall  f1-score   support

           0       0.87      0.68      0.76        19
           1       0.90      0.82      0.86        11
           2       0.70      0.84      0.76        19
           3       1.00      0.50      0.67         8
           4       0.80      0.80      0.80        15
           5       0.71      0.77      0.74        13
           6       0.56      0.47      0.51        19
           7       0.83      1.00      0.90        19
           8       0.61      0.58      0.59        19
           9       0.56      0.53      0.54        19
          10       1.00      0.62      0.77         8
          11       0.73      0.84      0.78        19
          12       1.00      0.50      0.67         8
          13       0.88      0.79      0.83        19
          14       0.68      0.68      0.68        19
          15       0.79      1.00      0.88        19
          16       0.89     

## Test case

In [24]:
test_sample = [
    "raise volume to level seven on music player", # "audio_volume_up"
    "which time zone are we in please change to current", # "datetime_convert"
    "show number and contact email of rehan", # "email_querycontact"
    "will you check and confirm the instruction please.", # "general_confirm"
    "is mile marker sixty five where the hanging tree is located", # "general_quirky"
    "brighten living room lights", # "iot_hue_lightup"
    "can you locate some gospel music for me", # "music_query"
    "how much is ten dollars in euros", # "qa_currency"
    "what is the capital of new hampshire", # "qa_factoid"
    "find updates from vicki's facebook from mardi gras day", # "social_query"
]

In [25]:
def decode_labels(indices, label_encoder):
    indices = np.array(indices).reshape(-1)
    return label_encoder.inverse_transform(indices)

In [26]:
tfidf_lr_test = tfidf_lr_pipeline.predict(test_sample)
decode_labels(tfidf_lr_test, le)

array(['audio_volume_up', 'datetime_convert', 'email_querycontact',
       'general_confirm', 'qa_factoid', 'iot_hue_lightup', 'play_music',
       'qa_currency', 'qa_factoid', 'social_query'], dtype=object)

In [27]:
test_sample_avg = np.array([sentence_to_avg_vector(text, w2v_model) for text in test_sample])

dense_test = np.argmax(dense_model.predict(test_sample_avg), axis=1)
decode_labels(dense_test, le)

1/1 [==============================] - 0s 20ms/step


array(['audio_volume_down', 'datetime_query', 'email_querycontact',
       'general_confirm', 'qa_factoid', 'iot_hue_lightup', 'play_music',
       'qa_currency', 'qa_factoid', 'social_query'], dtype=object)

In [28]:
test_sample_sequences = tokenizer.texts_to_sequences(test_sample)
test_sample_pad = pad_sequences(test_sample_sequences, maxlen=max_len, padding='post', truncating='post')

lstm_pretrained_test = np.argmax(lstm_model_pretrained.predict(test_sample_pad), axis=1)
decode_labels(lstm_pretrained_test, le)

1/1 [==============================] - 0s 50ms/step


array(['audio_volume_up', 'datetime_convert', 'email_querycontact',
       'general_confirm', 'qa_factoid', 'iot_hue_lightdim', 'play_music',
       'qa_currency', 'music_query', 'social_query'], dtype=object)

In [29]:
lstm_scratch_test = np.argmax(lstm_model_scratch.predict(test_sample_pad), axis=1)
decode_labels(lstm_scratch_test, le)

1/1 [==============================] - 0s 22ms/step


array(['audio_volume_up', 'datetime_query', 'email_querycontact',
       'general_confirm', 'recommendation_locations', 'iot_hue_lightup',
       'music_likeness', 'qa_currency', 'qa_factoid', 'social_query'],
      dtype=object)